# Model Adaptasi menggunakan dataset Live Traffic

Parameter:
1. Rasio split **70:15:15**.
2. Data train diseimbangkan menjadi **benign:attack = 1:1**.

Validation dan test tetap mengikuti distribusi natural live traffic. Dengan demikian, balancing hanya memengaruhi proses training, bukan proses kalibrasi threshold dan evaluasi akhir.


In [1]:
# ============================================================
# CELL 1 — SETUP
# ============================================================
from pathlib import Path
import os, json, zipfile, shutil, warnings, random
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print("Not running in Colab or Drive already mounted.")

try:
    import joblib
except Exception:
    import pickle as joblib

try:
    import tensorflow as tf
    from tensorflow import keras
    print("TensorFlow:", tf.__version__)
except Exception as e:
    raise RuntimeError("TensorFlow/Keras belum tersedia. Di Colab biasanya sudah tersedia.") from e

try:
    import lightgbm as lgb
    LIGHTGBM_AVAILABLE = True
    print("LightGBM:", lgb.__version__)
except Exception:
    LIGHTGBM_AVAILABLE = False
    raise ImportError("LightGBM belum tersedia. Jalankan: !pip -q install lightgbm lalu restart runtime.")

from sklearn.metrics import confusion_matrix, accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight

np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

Mounted at /content/drive
TensorFlow: 2.20.0
LightGBM: 4.6.0


In [ ]:
# ============================================================
# CELL 2 — FINAL CONFIG
# ============================================================

BASE_CICIDS = Path("/content/drive/MyDrive/CICIDS2018")
OUT_DIR = BASE_CICIDS / "ICICoS2026_FINAL_all_in_one_reviewer_compliant"

FINAL_DIR = OUT_DIR / "FINAL_V3D_BALANCED_SPLIT70_added_benign_bruteforce_live_target"
FINAL_MODEL_DIR = FINAL_DIR / "models"
FINAL_TABLE_DIR = FINAL_DIR / "tables"
FINAL_LOG_DIR = FINAL_DIR / "logs"

for d in [OUT_DIR, FINAL_DIR, FINAL_MODEL_DIR, FINAL_TABLE_DIR, FINAL_LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

MODEL_ORDER = ["AE-MLP", "MLP", "LightGBM"]

# Target CD2/live binary detection
TARGET_F1 = 0.85
TARGET_FPR = 0.02

# Split per skenario berdasarkan urutan timestamp.
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Untuk target live binary detection, default semua skenario live masuk train/val/test.
LIVE_HOLDOUT_UNSEEN_TEST_ONLY = []

# Training Keras
KERAS_LR = 1e-4
KERAS_EPOCHS = 60
KERAS_PATIENCE = 8
KERAS_BATCH = 256
BATCH_SIZE = 4096

# LightGBM
LGBM_NUM_BOOST_ROUND = 1200
LGBM_EARLY_STOPPING = 80

# Threshold sweep. Grid rapat agar bisa mencari FPR < 2%.
THRESHOLD_GRID = np.round(np.arange(0.01, 0.999, 0.001), 3)

# Calibration policies.
# Notebook akan memilih threshold dari validation set
VAL_FPR_TARGETS = [0.0, 0.005, 0.01, 0.015, 0.02]
TRAIN_FPR_GUARD_TARGETS = [0.0, 0.005, 0.01, 0.02]

# File input original dari training CSE-CIC-IDS2018
SCALER_PATH = BASE_CICIDS / "scaler.pkl"
AE_PATH = BASE_CICIDS / "ae_model_final.keras"
ENCODER_PATH = BASE_CICIDS / "ae_encoder_final.keras"
AE_CLF_V1_PATH = BASE_CICIDS / "ae_mlp_final.keras"
MLP_V1_PATH = BASE_CICIDS / "mlp_model_final.keras"
LGBM_V1_PATH = BASE_CICIDS / "lgbm_model_final.txt"

LIVE_ZIP_CANDIDATES = [
    BASE_CICIDS / "drive-download-20260629T134845Z-3-001.zip",
    BASE_CICIDS / "drive-download-20260629T134845Z-3-001.zip",
    BASE_CICIDS / "drive-download-20260623T132125Z-3-001.zip",
    BASE_CICIDS / "stageaware_live_logs.zip",
    BASE_CICIDS / "live_runtime_logs.zip",
]

LIVE_EXTRACT_DIR = OUT_DIR / "live_runtime_zip_extracted_FINAL_V3D_BALANCED_SPLIT70_added_benign_bruteforce"

USE_SCALER_FOR_KERAS = True
USE_SCALER_FOR_LIGHTGBM = True
EXPORT_TFLITE = True

print("BASE_CICIDS:", BASE_CICIDS)
print("FINAL_DIR:", FINAL_DIR)
print("Target F1:", TARGET_F1)
print("Target FPR:", TARGET_FPR)
print("LIVE_HOLDOUT_UNSEEN_TEST_ONLY:", LIVE_HOLDOUT_UNSEEN_TEST_ONLY)

BASE_CICIDS: /content/drive/MyDrive/CICIDS2018
FINAL_DIR: /content/drive/MyDrive/CICIDS2018/ICICoS2026_FINAL_all_in_one_reviewer_compliant/FINAL_V3D_BALANCED_SPLIT70_added_benign_bruteforce_live_target
Target F1: 0.85
Target FPR: 0.02
LIVE_HOLDOUT_UNSEEN_TEST_ONLY: []


## Catatan Metodologi Train Split 70:15:15

Notebook ini melakukan adaptasi ulang dari model awal hasil training CSE-CIC-IDS2018. Model yang dimuat adalah `ae_mlp_final.keras`, `mlp_model_final.keras`, dan `lgbm_model_final.txt`, lalu dilatih ulang menggunakan dataset live gabungan dari ZIP terbaru.

Rasio pembagian data diubah menjadi 70:15:15 agar porsi data latih lebih besar. Setelah split dilakukan, hanya data train yang diseimbangkan menjadi benign:attack = 1:1. Data validation dan test tetap mengikuti distribusi live traffic asli agar threshold calibration dan evaluasi akhir tetap merepresentasikan kondisi runtime sebenarnya.

Threshold dipilih dari validation set menggunakan kriteria F1-score tertinggi pada kandidat threshold yang memenuhi batas FPR. Test set hanya digunakan setelah threshold final dikunci.


In [3]:
# ============================================================
# CELL 3 — 38 FEATURE COLUMNS
# ============================================================

feature_cols = [
    'ACK_Flag_Cnt', 'Active_Max', 'Active_Min', 'Active_Std',
    'Bwd_Header_Len', 'Bwd_IAT_Max', 'Bwd_IAT_Mean', 'Bwd_IAT_Min',
    'Bwd_IAT_Std', 'Bwd_IAT_Tot', 'Bwd_Pkt_Len_Min', 'CWE_Flag_Count',
    'Down_Up_Ratio', 'FIN_Flag_Cnt', 'Flow_Byts_s', 'Flow_Duration',
    'Flow_IAT_Max', 'Flow_IAT_Std', 'Flow_Pkts_s', 'Fwd_Header_Len',
    'Fwd_IAT_Mean', 'Fwd_IAT_Std', 'Fwd_PSH_Flags', 'Fwd_Pkt_Len_Max',
    'Fwd_Pkt_Len_Mean', 'Fwd_Pkt_Len_Min', 'Fwd_Seg_Size_Min',
    'Idle_Max', 'Idle_Std', 'Init_Bwd_Win_Byts', 'Init_Fwd_Win_Byts',
    'PSH_Flag_Cnt', 'Pkt_Len_Max', 'Pkt_Len_Mean', 'Pkt_Len_Min',
    'Pkt_Len_Var', 'RST_Flag_Cnt', 'URG_Flag_Cnt'
]
assert len(feature_cols) == 38
print("Jumlah fitur:", len(feature_cols))

Jumlah fitur: 38


In [4]:
# ============================================================
# CELL 4 — FIND FILES
# ============================================================

def first_existing(paths, label="file", required=True):
    for p in paths:
        p = Path(p)
        if p.exists():
            print(f"{label} ditemukan:", p)
            return p
    if required:
        print(f"{label} tidak ditemukan. Kandidat:")
        for p in paths:
            print(" -", p)
        raise FileNotFoundError(f"{label} tidak ditemukan.")
    return None

def find_live_zip():
    for p in LIVE_ZIP_CANDIDATES:
        p = Path(p)
        if p.exists():
            print("LIVE ZIP ditemukan:", p)
            return p

    zips = list(BASE_CICIDS.glob("*.zip"))
    if not zips:
        raise FileNotFoundError(f"Tidak ada ZIP live log di {BASE_CICIDS}")

    preferred = [
        z for z in zips
        if any(k in z.name.lower() for k in ["20260629", "20260624", "new", "benign", "brute", "drive-download", "stage", "live", "runtime", "cc"])
    ]
    chosen = sorted(preferred if preferred else zips, key=lambda x: x.stat().st_mtime, reverse=True)[0]
    print("LIVE ZIP fallback terbaru:", chosen)
    return chosen

for p, label in [
    (SCALER_PATH, "scaler.pkl"),
    (AE_PATH, "ae_model_final.keras"),
    (ENCODER_PATH, "ae_encoder_final.keras"),
    (AE_CLF_V1_PATH, "ae_mlp_final.keras"),
    (MLP_V1_PATH, "mlp_model_final.keras"),
    (LGBM_V1_PATH, "lgbm_model_final.txt"),
]:
    if not Path(p).exists():
        raise FileNotFoundError(f"{label} tidak ditemukan: {p}")
    print(label, "OK:", p)

LIVE_ZIP_PATH = find_live_zip()

scaler.pkl OK: /content/drive/MyDrive/CICIDS2018/scaler.pkl
ae_model_final.keras OK: /content/drive/MyDrive/CICIDS2018/ae_model_final.keras
ae_encoder_final.keras OK: /content/drive/MyDrive/CICIDS2018/ae_encoder_final.keras
ae_mlp_final.keras OK: /content/drive/MyDrive/CICIDS2018/ae_mlp_final.keras
mlp_model_final.keras OK: /content/drive/MyDrive/CICIDS2018/mlp_model_final.keras
lgbm_model_final.txt OK: /content/drive/MyDrive/CICIDS2018/lgbm_model_final.txt
LIVE ZIP ditemukan: /content/drive/MyDrive/CICIDS2018/drive-download-20260629T134845Z-3-001.zip


In [ ]:
# ============================================================
# CELL 5 — LOAD LIVE IDS LOGS
# ============================================================

def normalize_model_name(raw):
    raw = str(raw).lower()
    if "aemlp" in raw or "ae-mlp" in raw:
        return "AE-MLP"
    if "lgbm" in raw or "lightgbm" in raw:
        return "LightGBM"
    if "mlp" in raw and "aemlp" not in raw and "ae-mlp" not in raw:
        return "MLP"
    return "Unknown"

def normalize_scenario_from_path(path):
    parts = [str(p).lower() for p in Path(path).parts]
    joined = "/".join(parts)

    if "benign" in joined:
        return "benign"
    if "slowhttp" in joined and "slowris" not in joined and "slowloris" not in joined:
        return "slowhttp"
    if "slowris" in joined or "slowloris" in joined:
        return "slowloris"
    if "bruteforce" in joined or "brute" in joined or "ssh" in joined:
        return "bruteforce"
    if "synflood" in joined or "hping" in joined or "/syn" in joined or "syn_" in joined:
        return "synflood"
    if "nmap" in joined or "portscan" in joined:
        return "nmap"
    if "nikto" in joined:
        return "nikto"
    if "gobuster" in joined:
        return "gobuster"
    return "unknown"

def normalize_columns(df):
    df = df.copy()
    df.columns = [str(c).strip().replace(" ", "_").replace("/", "_").replace("-", "_") for c in df.columns]
    return df

def find_matching_decision_log(ids_path):
    name = ids_path.name.replace("_ids_log.csv", "_decision_log.csv")
    candidate = ids_path.with_name(name)
    return candidate if candidate.exists() else None

if LIVE_EXTRACT_DIR.exists():
    shutil.rmtree(LIVE_EXTRACT_DIR)
LIVE_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(LIVE_ZIP_PATH, "r") as z:
    z.extractall(LIVE_EXTRACT_DIR)

print("Extracted to:", LIVE_EXTRACT_DIR)

all_csv = list(LIVE_EXTRACT_DIR.rglob("*.csv"))
def is_ids_log_file(path):
    name = path.name.lower()
    return ("ids_log" in name) and ("decision" not in name) and ("ips" not in name)

ids_files = [p for p in all_csv if is_ids_log_file(p)]
print("Jumlah CSV total:", len(all_csv))
print("Jumlah ids_log:", len(ids_files))

if not ids_files:
    raise RuntimeError("Tidak ditemukan *_ids_log.csv. Pastikan ZIP berisi log runtime fitur.")

records = []

for ids_path in ids_files:
    rel = ids_path.relative_to(LIVE_EXTRACT_DIR)
    rel_parts = rel.parts

    model_source = normalize_model_name(rel)
    scenario = normalize_scenario_from_path(rel)

    df = normalize_columns(pd.read_csv(ids_path))

    # Merge decision log jika ada; tidak wajib untuk target F1/FPR.
    dec_path = find_matching_decision_log(ids_path)
    if dec_path is not None:
        dec = normalize_columns(pd.read_csv(dec_path))
        key_cols = ["timestamp_us", "client_ip", "server_ip", "client_port", "server_port"]
        key_cols = [c for c in key_cols if c in df.columns and c in dec.columns]
        dec_cols = [c for c in [
            "timestamp_us", "client_ip", "server_ip", "client_port", "server_port",
            "block_threshold", "suspicious_threshold", "high_score_hit",
            "suspicious_hit", "block_decision", "ipset_applied", "decision", "action"
        ] if c in dec.columns]
        if len(key_cols) >= 3:
            df = df.merge(dec[dec_cols], on=key_cols, how="left", suffixes=("", "_decisionlog"))

    df["source_file"] = str(rel)
    df["log_model_source"] = model_source
    df["scenario"] = scenario
    df["ground_truth"] = 0 if scenario == "benign" else 1
    records.append(df)

live_df_all = pd.concat(records, ignore_index=True)

missing = [c for c in feature_cols if c not in live_df_all.columns]
if missing:
    print("Kolom tersedia:", list(live_df_all.columns))
    raise ValueError(f"Kolom fitur utama hilang: {missing}")

if "stage" not in live_df_all.columns:
    live_df_all["stage"] = "unknown"
live_df_all["stage"] = live_df_all["stage"].astype(str).str.lower().str.strip()

print("live_df_all:", live_df_all.shape)
display(live_df_all.groupby(["log_model_source", "scenario"]).size().rename("n_flow").reset_index())
display(live_df_all["scenario"].value_counts().rename_axis("scenario").reset_index(name="n_flow"))

Extracted to: /content/drive/MyDrive/CICIDS2018/ICICoS2026_FINAL_all_in_one_reviewer_compliant/live_runtime_zip_extracted_FINAL_V3D_BALANCED_SPLIT70_added_benign_bruteforce
Jumlah CSV total: 58
Jumlah ids_log: 20
live_df_all: (44437, 61)


,log_model_source,scenario,n_flow
0,AE-MLP,benign,1242
1,AE-MLP,bruteforce,243
2,AE-MLP,slowhttp,1061
3,AE-MLP,slowloris,1195
4,AE-MLP,synflood,5628
5,LightGBM,benign,1723
6,LightGBM,bruteforce,235
7,LightGBM,slowhttp,1052
8,LightGBM,slowloris,1176
9,LightGBM,synflood,5653


,scenario,n_flow
0,benign,17868
1,synflood,16556
2,slowloris,3515
3,bruteforce,3325
4,slowhttp,3173


In [6]:
# ============================================================
# CELL 6 — BUILD CANONICAL LIVE DATASET
# ============================================================

canonical_df = live_df_all.copy()

# Hapus scenario unknown agar label tidak salah.
unknown_count = int((canonical_df["scenario"] == "unknown").sum())
if unknown_count > 0:
    print("WARNING: ada scenario unknown, akan dikeluarkan:", unknown_count)
    display(canonical_df[canonical_df["scenario"] == "unknown"][["source_file"]].drop_duplicates().head(20))
    canonical_df = canonical_df[canonical_df["scenario"] != "unknown"].copy()

# Dedup exact. Timestamp/source berbeda tetap dianggap run berbeda.
dedup_keys = ["timestamp_us", "client_ip", "server_ip", "client_port", "server_port", "scenario"] + feature_cols
dedup_keys = [c for c in dedup_keys if c in canonical_df.columns]
before = len(canonical_df)
canonical_df = canonical_df.drop_duplicates(subset=dedup_keys).reset_index(drop=True)
after = len(canonical_df)

print("Canonical live dataset:", canonical_df.shape)
print("Removed exact duplicates:", before - after)

scenario_counts = canonical_df["scenario"].value_counts().rename_axis("scenario").reset_index(name="n_flow")
label_counts = canonical_df["ground_truth"].value_counts().rename_axis("label").reset_index(name="n_flow")

display(scenario_counts)
display(label_counts)

scenario_counts.to_csv(FINAL_TABLE_DIR / "FINAL_V3D_BALANCED_SPLIT70_dataset_scenario_counts.csv", index=False)
label_counts.to_csv(FINAL_TABLE_DIR / "FINAL_V3D_BALANCED_SPLIT70_dataset_label_counts.csv", index=False)
canonical_df.to_csv(FINAL_LOG_DIR / "FINAL_V3D_BALANCED_SPLIT70_canonical_dataset.csv", index=False)

print("Saved canonical dataset and counts.")

Canonical live dataset: (44437, 61)
Removed exact duplicates: 0


,scenario,n_flow
0,benign,17868
1,synflood,16556
2,slowloris,3515
3,bruteforce,3325
4,slowhttp,3173


,label,n_flow
0,1,26569
1,0,17868


Saved canonical dataset and counts.


In [7]:
# ============================================================
# CELL 7 — TRAIN / VALIDATION / TEST SPLIT
# ============================================================

def time_order_split_by_scenario(df):
    train_parts, val_parts, test_parts = [], [], []
    split_rows = []

    for scenario, g in df.groupby("scenario"):
        g = g.sort_values("timestamp_us").reset_index(drop=True)
        n = len(g)

        if scenario in LIVE_HOLDOUT_UNSEEN_TEST_ONLY:
            g = g.copy()
            g["split"] = "test_unseen"
            test_parts.append(g)
            split_rows.append({
                "scenario": scenario, "n_total": n, "train": 0, "val": 0, "test": n,
                "policy": "held out as unseen test only"
            })
            continue

        if n < 10:
            g = g.copy()
            g["split"] = "test_small"
            test_parts.append(g)
            split_rows.append({
                "scenario": scenario, "n_total": n, "train": 0, "val": 0, "test": n,
                "policy": "too small, test only"
            })
            continue

        i1 = int(TRAIN_RATIO * n)
        i2 = int((TRAIN_RATIO + VAL_RATIO) * n)

        train = g.iloc[:i1].copy()
        val = g.iloc[i1:i2].copy()
        test = g.iloc[i2:].copy()

        train["split"] = "train"
        val["split"] = "val"
        test["split"] = "test"

        train_parts.append(train)
        val_parts.append(val)
        test_parts.append(test)

        split_rows.append({
            "scenario": scenario,
            "n_total": n,
            "train": len(train),
            "val": len(val),
            "test": len(test),
            "policy": "time-ordered 70/15/15"
        })

    train_df = pd.concat(train_parts, ignore_index=True) if train_parts else pd.DataFrame()
    val_df = pd.concat(val_parts, ignore_index=True) if val_parts else pd.DataFrame()
    test_df = pd.concat(test_parts, ignore_index=True) if test_parts else pd.DataFrame()
    split_df = pd.DataFrame(split_rows).sort_values("scenario")

    return train_df, val_df, test_df, split_df

train_df, val_df, test_df, split_df = time_order_split_by_scenario(canonical_df)

display(split_df)

print("Train:", train_df.shape, train_df["ground_truth"].value_counts().to_dict())
print("Val:", val_df.shape, val_df["ground_truth"].value_counts().to_dict())
print("Test:", test_df.shape, test_df["ground_truth"].value_counts().to_dict())

split_df.to_csv(FINAL_TABLE_DIR / "FINAL_V3D_BALANCED_SPLIT70_split_summary.csv", index=False)
train_df.to_csv(FINAL_LOG_DIR / "FINAL_V3D_BALANCED_SPLIT70_train.csv", index=False)
val_df.to_csv(FINAL_LOG_DIR / "FINAL_V3D_BALANCED_SPLIT70_val.csv", index=False)
test_df.to_csv(FINAL_LOG_DIR / "FINAL_V3D_BALANCED_SPLIT70_test.csv", index=False)

print("Saved split files.")

,scenario,n_total,train,val,test,policy
0,benign,17868,12507,2680,2681,time-ordered 70/15/15
1,bruteforce,3325,2327,499,499,time-ordered 70/15/15
2,slowhttp,3173,2221,476,476,time-ordered 70/15/15
3,slowloris,3515,2460,527,528,time-ordered 70/15/15
4,synflood,16556,11589,2483,2484,time-ordered 70/15/15


Train: (31104, 62) {1: 18597, 0: 12507}
Val: (6665, 62) {1: 3985, 0: 2680}
Test: (6668, 62) {1: 3987, 0: 2681}
Saved split files.


## Balanced Training Policy

Cell berikut hanya menyeimbangkan **data train** menjadi benign:attack = 1:1. Data validation dan test **tidak diseimbangkan** agar proses kalibrasi threshold dan evaluasi akhir tetap mengikuti distribusi live traffic asli. Attack diambil hampir merata dari setiap skenario agar data latih tidak terlalu didominasi oleh SYN flood.


In [8]:
# ============================================================
# CELL 7B — BALANCED TRAINING SET ONLY
# ============================================================
# Tujuan:
# - Data train dibuat seimbang: benign : attack = 1 : 1.
# - Validation dan test TIDAK diseimbangkan agar evaluasi tetap natural sesuai live traffic.
# - Sampel attack dipilih hampir merata per skenario agar tidak didominasi SYN flood.

BALANCE_TRAIN_ONLY = True
BALANCED_RANDOM_STATE = 42
BALANCED_ATTACK_TO_BENIGN_RATIO = 1.0

train_df_natural = train_df.copy().reset_index(drop=True)
val_df_natural = val_df.copy().reset_index(drop=True)
test_df_natural = test_df.copy().reset_index(drop=True)

def allocate_evenly_with_caps(counts, total_target):
    counts = {k: int(v) for k, v in counts.items()}
    keys = sorted(counts.keys())
    alloc = {k: 0 for k in keys}
    remaining = int(total_target)
    active = set(keys)

    while remaining > 0 and active:
        quota = max(1, remaining // len(active))
        progressed = False
        for k in list(sorted(active)):
            spare = counts[k] - alloc[k]
            if spare <= 0:
                active.remove(k)
                continue
            add = min(quota, spare, remaining)
            alloc[k] += add
            remaining -= add
            progressed = True
            if alloc[k] >= counts[k]:
                active.remove(k)
            if remaining <= 0:
                break
        if not progressed:
            break
    return alloc

def make_balanced_train(df, random_state=42):
    benign = df[df["ground_truth"] == 0].copy()
    attack = df[df["ground_truth"] == 1].copy()

    n_benign = len(benign)
    target_attack = min(len(attack), int(round(n_benign * BALANCED_ATTACK_TO_BENIGN_RATIO)))

    attack_counts = attack["scenario"].value_counts().to_dict()
    attack_alloc = allocate_evenly_with_caps(attack_counts, target_attack)

    sampled_attack_parts = []
    for scenario, n_take in attack_alloc.items():
        if n_take <= 0:
            continue
        g = attack[attack["scenario"] == scenario]
        sampled_attack_parts.append(g.sample(n=n_take, random_state=random_state, replace=False))

    sampled_attack = pd.concat(sampled_attack_parts, ignore_index=True) if sampled_attack_parts else attack.iloc[0:0].copy()
    balanced = pd.concat([benign, sampled_attack], ignore_index=True)
    balanced = balanced.sample(frac=1.0, random_state=random_state).reset_index(drop=True)
    balanced["split"] = "train_balanced"

    summary_before = df.groupby(["scenario", "ground_truth"]).size().reset_index(name="n_before")
    summary_after = balanced.groupby(["scenario", "ground_truth"]).size().reset_index(name="n_after")
    summary = summary_before.merge(summary_after, on=["scenario", "ground_truth"], how="left").fillna({"n_after": 0})
    summary["n_after"] = summary["n_after"].astype(int)

    return balanced, summary, attack_alloc

if BALANCE_TRAIN_ONLY:
    train_df_balanced, balance_summary_df, attack_alloc = make_balanced_train(
        train_df_natural,
        random_state=BALANCED_RANDOM_STATE,
    )
    train_df = train_df_balanced.copy().reset_index(drop=True)
else:
    train_df_balanced = train_df_natural.copy().reset_index(drop=True)
    balance_summary_df = train_df_balanced.groupby(["scenario", "ground_truth"]).size().reset_index(name="n_after")
    attack_alloc = "not applied"

print("Natural train label distribution:", train_df_natural["ground_truth"].value_counts().sort_index().to_dict())
print("Balanced train label distribution:", train_df["ground_truth"].value_counts().sort_index().to_dict())
print("Validation tetap natural:", val_df["ground_truth"].value_counts().sort_index().to_dict())
print("Test tetap natural:", test_df["ground_truth"].value_counts().sort_index().to_dict())
print("Attack allocation in balanced train:", attack_alloc)

display(balance_summary_df.sort_values(["ground_truth", "scenario"]))

# Simpan bukti distribusi agar bisa dipakai untuk Bab 4/Bab 5.
train_df_natural.to_csv(FINAL_LOG_DIR / "FINAL_V3D_BALANCED_SPLIT70_train_natural_before_balance.csv", index=False)
train_df_balanced.to_csv(FINAL_LOG_DIR / "FINAL_V3D_BALANCED_SPLIT70_train_balanced_only.csv", index=False)
balance_summary_df.to_csv(FINAL_TABLE_DIR / "FINAL_V3D_BALANCED_SPLIT70_train_sampling_summary.csv", index=False)

print("Saved natural train, balanced train, and sampling summary.")


Natural train label distribution: {0: 12507, 1: 18597}
Balanced train label distribution: {0: 12507, 1: 12507}
Validation tetap natural: {0: 2680, 1: 3985}
Test tetap natural: {0: 2681, 1: 3987}
Attack allocation in balanced train: {'bruteforce': 2327, 'slowhttp': 2221, 'slowloris': 2460, 'synflood': 5499}


,scenario,ground_truth,n_before,n_after
0,benign,0,12507,12507
1,bruteforce,1,2327,2327
2,slowhttp,1,2221,2221
3,slowloris,1,2460,2460
4,synflood,1,11589,5499


Saved natural train, balanced train, and sampling summary.


In [9]:
# ============================================================
# CELL 8 — LOAD SCALER AND ORIGINAL V1 MODELS
# ============================================================

scaler = joblib.load(SCALER_PATH)
print("Scaler loaded:", type(scaler))

ae_model = keras.models.load_model(AE_PATH, compile=False)
ae_encoder = keras.models.load_model(ENCODER_PATH, compile=False)
ae_classifier_v1 = keras.models.load_model(AE_CLF_V1_PATH, compile=False)
mlp_model_v1 = keras.models.load_model(MLP_V1_PATH, compile=False)
lgbm_model_v1 = lgb.Booster(model_file=str(LGBM_V1_PATH))

print("AE model:", ae_model.input_shape, ae_model.output_shape)
print("AE encoder:", ae_encoder.input_shape, ae_encoder.output_shape)
print("AE-MLP classifier V1:", ae_classifier_v1.input_shape, ae_classifier_v1.output_shape)
print("MLP V1:", mlp_model_v1.input_shape, mlp_model_v1.output_shape)
print("LightGBM V1 loaded.")

Scaler loaded: <class 'sklearn.preprocessing._data.QuantileTransformer'>
AE model: (None, 38) (None, 38)
AE encoder: (None, 38) (None, 8)
AE-MLP classifier V1: (None, 9) (None, 1)
MLP V1: (None, 38) (None, 1)
LightGBM V1 loaded.


In [10]:
# ============================================================
# CELL 9 — HELPERS
# ============================================================

def clean_X(df):
    X = df[feature_cols].copy()
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return X.astype("float32").values

def y_of(df):
    return df["ground_truth"].astype(int).values

def transform_X(X, use_scaler=True):
    if use_scaler and scaler is not None:
        return scaler.transform(X).astype("float32")
    return X.astype("float32")

def _as_1d_score(pred):
    arr = np.asarray(pred)
    if arr.ndim == 1:
        return arr.astype("float64")
    if arr.ndim == 2 and arr.shape[1] == 1:
        return arr[:, 0].astype("float64")
    if arr.ndim == 2 and arr.shape[1] >= 2:
        return arr[:, 1].astype("float64")
    return arr.reshape(-1).astype("float64")

def make_ae_classifier_input(X_scaled):
    latent = ae_encoder.predict(X_scaled, batch_size=BATCH_SIZE, verbose=0)
    recon = ae_model.predict(X_scaled, batch_size=BATCH_SIZE, verbose=0)
    recon_error = np.mean(np.square(X_scaled - recon), axis=1, keepdims=True)

    clf_dim = int(ae_classifier_v1.input_shape[-1])
    latent_dim = int(latent.shape[1])
    raw_dim = int(X_scaled.shape[1])

    if clf_dim == latent_dim + 1:
        return np.concatenate([latent, recon_error], axis=1), f"latent({latent_dim}) + recon_error"
    if clf_dim == latent_dim:
        return latent, f"latent({latent_dim})"
    if clf_dim == raw_dim + 1:
        return np.concatenate([X_scaled, recon_error], axis=1), f"scaled_raw({raw_dim}) + recon_error"
    if clf_dim == raw_dim:
        return X_scaled, f"scaled_raw({raw_dim})"

    raise ValueError(f"AE classifier input dim tidak cocok: {clf_dim}")

def class_weight_dict(y):
    classes = np.array([0, 1])
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y.astype(int))
    return {int(c): float(w) for c, w in zip(classes, weights)}

def binary_metrics(y_true, scores, threshold):
    y_pred = (scores >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    return {
        "n": int(len(y_true)),
        "threshold": float(threshold),
        "accuracy": float(acc),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "fpr": float(fpr),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

def predict_scores(model_name, model_obj, df):
    X_raw = clean_X(df)

    if model_name == "AE-MLP":
        Xs = transform_X(X_raw, USE_SCALER_FOR_KERAS)
        Z, input_used = make_ae_classifier_input(Xs)
        scores = _as_1d_score(model_obj.predict(Z, batch_size=BATCH_SIZE, verbose=0))
        return scores, input_used

    if model_name == "MLP":
        Xs = transform_X(X_raw, USE_SCALER_FOR_KERAS)
        scores = _as_1d_score(model_obj.predict(Xs, batch_size=BATCH_SIZE, verbose=0))
        return scores, "scaled_raw" if USE_SCALER_FOR_KERAS else "raw"

    if model_name == "LightGBM":
        Xs = transform_X(X_raw, USE_SCALER_FOR_LIGHTGBM)
        scores = _as_1d_score(model_obj.predict(Xs))
        return scores, "scaled_raw" if USE_SCALER_FOR_LIGHTGBM else "raw"

    raise ValueError(model_name)

def export_tflite_model(model, out_path):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    tflite_model = converter.convert()
    Path(out_path).write_bytes(tflite_model)
    return out_path

def scenario_table(df, scores, threshold, model, version):
    tmp = df[["scenario", "ground_truth"]].copy()
    tmp["score"] = scores
    tmp["pred"] = (scores >= threshold).astype(int)

    rows = []
    for scenario, g in tmp.groupby("scenario"):
        y = g["ground_truth"].astype(int).values
        s = g["score"].values
        pred = g["pred"].astype(int).values

        if len(np.unique(y)) == 1 and y[0] == 0:
            fp = int((pred == 1).sum())
            tn = int((pred == 0).sum())
            rows.append({
                "model": model, "version": version, "scenario": scenario, "ground_truth": 0,
                "n_flow": int(len(g)), "score_mean": float(s.mean()), "score_min": float(s.min()), "score_max": float(s.max()),
                "pred_benign": int(tn), "pred_attack": int(fp),
                "fpr": float(fp / (fp + tn)) if (fp + tn) else 0.0,
                "detection_rate": np.nan,
            })
        elif len(np.unique(y)) == 1 and y[0] == 1:
            tp = int((pred == 1).sum())
            fn = int((pred == 0).sum())
            rows.append({
                "model": model, "version": version, "scenario": scenario, "ground_truth": 1,
                "n_flow": int(len(g)), "score_mean": float(s.mean()), "score_min": float(s.min()), "score_max": float(s.max()),
                "pred_benign": int(fn), "pred_attack": int(tp),
                "fpr": np.nan,
                "detection_rate": float(tp / (tp + fn)) if (tp + fn) else 0.0,
            })
        else:
            m = binary_metrics(y, s, threshold)
            rows.append({
                "model": model, "version": version, "scenario": scenario, "ground_truth": -1,
                "n_flow": int(len(g)), "score_mean": float(s.mean()), "score_min": float(s.min()), "score_max": float(s.max()),
                "pred_benign": int((pred == 0).sum()), "pred_attack": int((pred == 1).sum()),
                "fpr": m["fpr"],
                "detection_rate": m["recall"],
            })

    return pd.DataFrame(rows)

print("Helpers ready.")

Helpers ready.


In [11]:
# ============================================================
# CELL 10 — PREPARE ARRAYS
# ============================================================

X_train_raw, y_train = clean_X(train_df), y_of(train_df)
X_val_raw, y_val = clean_X(val_df), y_of(val_df)
X_test_raw, y_test = clean_X(test_df), y_of(test_df)

X_train = transform_X(X_train_raw, USE_SCALER_FOR_KERAS)
X_val = transform_X(X_val_raw, USE_SCALER_FOR_KERAS)
X_test = transform_X(X_test_raw, USE_SCALER_FOR_KERAS)

print("X_train:", X_train.shape, "y:", np.bincount(y_train))
print("X_val:", X_val.shape, "y:", np.bincount(y_val))
print("X_test:", X_test.shape, "y:", np.bincount(y_test))

X_train: (25014, 38) y: [12507 12507]
X_val: (6665, 38) y: [2680 3985]
X_test: (6668, 38) y: [2681 3987]


In [12]:
# ============================================================
# CELL 11 — TRAIN AE-MLP V3D CLASSIFIER
# ============================================================

print("Preparing AE-MLP classifier input...")
Z_train, ae_input_used = make_ae_classifier_input(X_train)
Z_val, _ = make_ae_classifier_input(X_val)
Z_test, _ = make_ae_classifier_input(X_test)

print("AE classifier input:", ae_input_used)
print("Z_train:", Z_train.shape, "Z_val:", Z_val.shape, "Z_test:", Z_test.shape)

ae_classifier_v3c = keras.models.clone_model(ae_classifier_v1)
ae_classifier_v3c.set_weights(ae_classifier_v1.get_weights())

ae_classifier_v3c.compile(
    optimizer=keras.optimizers.Adam(learning_rate=KERAS_LR),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

cw = class_weight_dict(y_train)
print("Class weight:", cw)

ae_hist = ae_classifier_v3c.fit(
    Z_train, y_train,
    validation_data=(Z_val, y_val),
    epochs=KERAS_EPOCHS,
    batch_size=KERAS_BATCH,
    class_weight=cw,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=KERAS_PATIENCE, restore_best_weights=True)
    ],
    verbose=1
)

ae_v3c_path = FINAL_MODEL_DIR / "ae_mlp_classifier_v3d_balanced_train_split_70_15_15_f1fpr_added_benign_bruteforce.keras"
ae_classifier_v3c.save(ae_v3c_path)
print("Saved:", ae_v3c_path)

if EXPORT_TFLITE:
    ae_tflite_path = FINAL_MODEL_DIR / "ae_mlp_classifier_v3d_balanced_train_split_70_15_15_f1fpr_added_benign_bruteforce.tflite"
    export_tflite_model(ae_classifier_v3c, ae_tflite_path)
    print("Saved:", ae_tflite_path)

Preparing AE-MLP classifier input...
AE classifier input: latent(8) + recon_error
Z_train: (25014, 9) Z_val: (6665, 9) Z_test: (6668, 9)
Class weight: {0: 1.0, 1: 1.0}
Epoch 1/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.5123 - loss: 3.1830 - val_accuracy: 0.7077 - val_loss: 0.6757
Epoch 2/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.5302 - loss: 1.8000 - val_accuracy: 0.7737 - val_loss: 0.6187
Epoch 3/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.5612 - loss: 1.1852 - val_accuracy: 0.7337 - val_loss: 0.6231
Epoch 4/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.6001 - loss: 0.9366 - val_accuracy: 0.7661 - val_loss: 0.6066
Epoch 5/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6406 - loss: 0.8088 - val_accuracy: 0.7515 - val_loss: 0.6035
Epoch 6/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6651 - loss: 0.7404 - val_accuracy: 0.7958 - val_loss: 0.5897
Epoch 7/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6767 - loss:

In [13]:
# ============================================================
# CELL 12 — TRAIN MLP V3D
# ============================================================

mlp_model_v3c = keras.models.clone_model(mlp_model_v1)
mlp_model_v3c.set_weights(mlp_model_v1.get_weights())

mlp_model_v3c.compile(
    optimizer=keras.optimizers.Adam(learning_rate=KERAS_LR),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

cw = class_weight_dict(y_train)
print("Class weight:", cw)

mlp_hist = mlp_model_v3c.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=KERAS_EPOCHS,
    batch_size=KERAS_BATCH,
    class_weight=cw,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=KERAS_PATIENCE, restore_best_weights=True)
    ],
    verbose=1
)

mlp_v3c_path = FINAL_MODEL_DIR / "mlp_model_v3d_balanced_train_split_70_15_15_f1fpr_added_benign_bruteforce.keras"
mlp_model_v3c.save(mlp_v3c_path)
print("Saved:", mlp_v3c_path)

if EXPORT_TFLITE:
    mlp_tflite_path = FINAL_MODEL_DIR / "mlp_model_v3d_balanced_train_split_70_15_15_f1fpr_added_benign_bruteforce.tflite"
    export_tflite_model(mlp_model_v3c, mlp_tflite_path)
    print("Saved:", mlp_tflite_path)

Class weight: {0: 1.0, 1: 1.0}
Epoch 1/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - accuracy: 0.5912 - loss: 3.2569 - val_accuracy: 0.4021 - val_loss: 7.7324
Epoch 2/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.6854 - loss: 1.6908 - val_accuracy: 0.6344 - val_loss: 1.8789
Epoch 3/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.7243 - loss: 1.0821 - val_accuracy: 0.7917 - val_loss: 0.7259
Epoch 4/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.7432 - loss: 0.7974 - val_accuracy: 0.8416 - val_loss: 0.4605
Epoch 5/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.7654 - loss: 0.6294 - val_accuracy: 0.8477 - val_loss: 0.3641
Epoch 6/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.7734 - loss: 0.5381 - val_accuracy: 0.8552 - val_loss: 0.3280
Epoch 7/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 37ms/step - accuracy: 0.7730 - loss: 0.5014 - val_accuracy: 0.8560 - val_loss: 0.3211
Epoch 8/60
98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - accuracy: 0.7806 - loss: 0.

In [14]:
# ============================================================
# CELL 13 — TRAIN LIGHTGBM V3D
# ============================================================

pos = int((y_train == 1).sum())
neg = int((y_train == 0).sum())
scale_pos_weight = neg / pos if pos > 0 else 1.0
print("neg:", neg, "pos:", pos, "scale_pos_weight:", scale_pos_weight)

X_train_lgb = transform_X(X_train_raw, USE_SCALER_FOR_LIGHTGBM)
X_val_lgb = transform_X(X_val_raw, USE_SCALER_FOR_LIGHTGBM)

dtrain = lgb.Dataset(X_train_lgb, label=y_train, feature_name=feature_cols)
dval = lgb.Dataset(X_val_lgb, label=y_val, feature_name=feature_cols, reference=dtrain)

params = {
    "objective": "binary",
    "metric": ["binary_logloss"],
    "learning_rate": 0.03,
    "num_leaves": 31,
    "max_depth": 6,
    "min_data_in_leaf": 10,
    "feature_fraction": 0.95,
    "bagging_fraction": 0.95,
    "bagging_freq": 1,
    "lambda_l1": 0.0,
    "lambda_l2": 1.0,
    "scale_pos_weight": scale_pos_weight,
    "verbosity": -1,
    "seed": 42,
}

lgbm_model_v3c = lgb.train(
    params,
    dtrain,
    valid_sets=[dtrain, dval],
    valid_names=["train", "val"],
    num_boost_round=LGBM_NUM_BOOST_ROUND,
    callbacks=[
        lgb.early_stopping(LGBM_EARLY_STOPPING),
        lgb.log_evaluation(50),
    ],
)

lgbm_v3c_path = FINAL_MODEL_DIR / "lgbm_model_v3d_balanced_train_split_70_15_15_f1fpr_added_benign_bruteforce.txt"
lgbm_model_v3c.save_model(str(lgbm_v3c_path))
print("Saved:", lgbm_v3c_path)

neg: 12507 pos: 12507 scale_pos_weight: 1.0
Training until validation scores don't improve for 80 rounds
[50]	train's binary_logloss: 0.323414	val's binary_logloss: 0.280072
[100]	train's binary_logloss: 0.26715	val's binary_logloss: 0.222292
[150]	train's binary_logloss: 0.253943	val's binary_logloss: 0.211101
[200]	train's binary_logloss: 0.248152	val's binary_logloss: 0.208787
[250]	train's binary_logloss: 0.243075	val's binary_logloss: 0.208248
[300]	train's binary_logloss: 0.236554	val's binary_logloss: 0.207411
[350]	train's binary_logloss: 0.23081	val's binary_logloss: 0.207311
[400]	train's binary_logloss: 0.225741	val's binary_logloss: 0.207078
[450]	train's binary_logloss: 0.221645	val's binary_logloss: 0.206998
Early stopping, best iteration is:
[418]	train's binary_logloss: 0.224127	val's binary_logloss: 0.206849
Saved: /content/drive/MyDrive/CICIDS2018/ICICoS2026_FINAL_all_in_one_reviewer_compliant/FINAL_V3D_BALANCED_SPLIT70_added_benign_bruteforce_live_target/models/lgbm_

In [15]:
# ============================================================
# CELL 14 — PRECOMPUTE SCORES FOR THRESHOLD CALIBRATION
# ============================================================

models_v3c = {
    "AE-MLP": ae_classifier_v3c,
    "MLP": mlp_model_v3c,
    "LightGBM": lgbm_model_v3c,
}

score_cache = {}

for model_name in MODEL_ORDER:
    model_obj = models_v3c[model_name]
    for split_name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        scores, input_used = predict_scores(model_name, model_obj, df)
        score_cache[(model_name, split_name)] = {
            "scores": scores,
            "input_used": input_used,
        }
        print(model_name, split_name, len(scores), input_used)

print("Score cache ready.")

AE-MLP train 25014 latent(8) + recon_error
AE-MLP val 6665 latent(8) + recon_error
AE-MLP test 6668 latent(8) + recon_error
MLP train 25014 scaled_raw
MLP val 6665 scaled_raw
MLP test 6668 scaled_raw
LightGBM train 25014 scaled_raw
LightGBM val 6665 scaled_raw
LightGBM test 6668 scaled_raw
Score cache ready.


In [16]:
# ============================================================
# CELL 15 — F1/FPR THRESHOLD CALIBRATION
# ============================================================
# Threshold dipilih dari validation set, bukan test set.
# Objective utama: val_f1 tertinggi.
# Constraint utama: val_fpr sesuai target policy dan train_fpr_guard.

policy_rows = []
grid_all_parts = []

for model_name in MODEL_ORDER:
    s_train = score_cache[(model_name, "train")]["scores"]
    s_val = score_cache[(model_name, "val")]["scores"]
    s_test = score_cache[(model_name, "test")]["scores"]
    input_used = score_cache[(model_name, "test")]["input_used"]

    for val_fpr_target in VAL_FPR_TARGETS:
        for train_fpr_guard in TRAIN_FPR_GUARD_TARGETS:
            rows = []
            for th in THRESHOLD_GRID:
                m_train = binary_metrics(y_train, s_train, th)
                m_val = binary_metrics(y_val, s_val, th)
                rows.append({
                    "model": model_name,
                    "input_used": input_used,
                    "selection_objective": "val_f1_with_fpr_constraint",
                    "val_fpr_target": float(val_fpr_target),
                    "train_fpr_guard": float(train_fpr_guard),
                    "threshold": float(th),
                    "train_f1": m_train["f1"],
                    "train_fpr": m_train["fpr"],
                    "train_precision": m_train["precision"],
                    "train_recall": m_train["recall"],
                    "val_f1": m_val["f1"],
                    "val_fpr": m_val["fpr"],
                    "val_precision": m_val["precision"],
                    "val_recall": m_val["recall"],
                    "val_tn": m_val["tn"], "val_fp": m_val["fp"], "val_fn": m_val["fn"], "val_tp": m_val["tp"],
                })

            grid = pd.DataFrame(rows)
            grid_all_parts.append(grid)

            feasible = grid[
                (grid["val_fpr"] <= val_fpr_target) &
                (grid["train_fpr"] <= train_fpr_guard)
            ].copy()

            if feasible.empty:
                continue

            chosen = feasible.sort_values(
                ["val_f1", "val_fpr", "val_recall", "val_precision", "threshold"],
                ascending=[False, True, False, False, True]
            ).iloc[0].to_dict()

            test_m = binary_metrics(y_test, s_test, chosen["threshold"])

            out = chosen.copy()
            out.update({
                "test_accuracy": test_m["accuracy"],
                "test_precision": test_m["precision"],
                "test_recall": test_m["recall"],
                "test_f1": test_m["f1"],
                "test_fpr": test_m["fpr"],
                "test_tn": test_m["tn"], "test_fp": test_m["fp"], "test_fn": test_m["fn"], "test_tp": test_m["tp"],
                "pass_f1": bool(test_m["f1"] >= TARGET_F1),
                "pass_fpr": bool(test_m["fpr"] < TARGET_FPR),
                "pass_both": bool((test_m["f1"] >= TARGET_F1) and (test_m["fpr"] < TARGET_FPR)),
            })
            policy_rows.append(out)

policy_df = pd.DataFrame(policy_rows)
grid_all_df = pd.concat(grid_all_parts, ignore_index=True)

display(policy_df.sort_values(["model", "val_f1", "val_fpr", "test_f1"], ascending=[True, False, True, False]))
policy_df.to_csv(FINAL_TABLE_DIR / "FINAL_V3D_BALANCED_SPLIT70_f1fpr_threshold_policy_sweep.csv", index=False)
grid_all_df.to_csv(FINAL_TABLE_DIR / "FINAL_V3D_BALANCED_SPLIT70_all_threshold_grid.csv", index=False)

print("Saved F1/FPR threshold sweep tables.")


,model,input_used,selection_objective,val_fpr_target,train_fpr_guard,threshold,train_f1,train_fpr,train_precision,train_recall,...,test_recall,test_f1,test_fpr,test_tn,test_fp,test_fn,test_tp,pass_f1,pass_fpr,pass_both
5,AE-MLP,latent(8) + recon_error,val_f1_with_fpr_constraint,0.005,0.005,0.683,0.791665,0.001439,0.997811,0.656113,...,0.835214,0.910085,0.000373,2680,1,657,3330,True,True,True
6,AE-MLP,latent(8) + recon_error,val_f1_with_fpr_constraint,0.005,0.010,0.683,0.791665,0.001439,0.997811,0.656113,...,0.835214,0.910085,0.000373,2680,1,657,3330,True,True,True
7,AE-MLP,latent(8) + recon_error,val_f1_with_fpr_constraint,0.005,0.020,0.683,0.791665,0.001439,0.997811,0.656113,...,0.835214,0.910085,0.000373,2680,1,657,3330,True,True,True
9,AE-MLP,latent(8) + recon_error,val_f1_with_fpr_constraint,0.010,0.005,0.683,0.791665,0.001439,0.997811,0.656113,...,0.835214,0.910085,0.000373,2680,1,657,3330,True,True,True
10,AE-MLP,latent(8) + recon_error,val_f1_with_fpr_constraint,0.010,0.010,0.683,0.791665,0.001439,0.997811,0.656113,...,0.835214,0.910085,0.000373,2680,1,657,3330,True,True,True
11,AE-MLP,latent(8) + recon_error,val_f1_with_fpr_constraint,0.010,0.020,0.683,0.791665,0.001439,0.997811,0.656113,...,0.835214,0.910085,0.000373,2680,1,657,3330,True,True,True
13,AE-MLP,latent(8) + recon_error,val_f1_with_fpr_constraint,0.015,0.005,0.683,0.791665,0.001439,0.997811,0.656113,...,0.835214,0.910085,0.000373,2680,1,657,3330,True,True,True
14,AE-MLP,latent(8) + recon_error,val_f1_with_fpr_constraint,0.015,0.010,0.683,0.791665,0.001439,0.997811,0.656113,...,0.835214,0.910085,0.000373,2680,1,657,3330,True,True,True
15,AE-MLP,latent(8) + recon_error,val_f1_with_fpr_constraint,0.015,0.020,0.683,0.791665,0.001439,0.997811,0.656113,...,0.835214,0.910085,0.000373,2680,1,657,3330,True,True,True
17,AE-MLP,latent(8) + recon_error,val_f1_with_fpr_constraint,0.020,0.005,0.683,0.791665,0.001439,0.997811,0.656113,...,0.835214,0.910085,0.000373,2680,1,657,3330,True,True,True


Saved F1/FPR threshold sweep tables.


In [17]:
# ============================================================
# CELL 16 — FINAL MODEL SELECTION AND EVALUATION
# ============================================================
# Best threshold per model dipilih dari validation metrics.
# Test metrics hanya dihitung setelah threshold final dipilih.

final_rows = []
scenario_parts = []

for model_name in MODEL_ORDER:
    sub = policy_df[policy_df["model"] == model_name].copy()

    if sub.empty:
        print("No feasible threshold for", model_name)
        continue

    strict = sub[sub["val_fpr_target"] <= TARGET_FPR].copy()
    candidates = strict if not strict.empty else sub

    best = candidates.sort_values(
        ["val_f1", "val_fpr", "val_recall", "val_precision", "threshold"],
        ascending=[False, True, False, False, True]
    ).iloc[0]

    th = float(best["threshold"])
    s_test = score_cache[(model_name, "test")]["scores"]
    m_test = binary_metrics(y_test, s_test, th)

    if (m_test["f1"] >= TARGET_F1) and (m_test["fpr"] < TARGET_FPR):
        final_status = "PASS_BOTH"
    elif m_test["fpr"] < TARGET_FPR:
        final_status = "PASS_FPR_ONLY"
    elif m_test["f1"] >= TARGET_F1:
        final_status = "PASS_F1_ONLY"
    else:
        final_status = "NOT_PASS_TARGET"

    row = {
        "model": model_name,
        "version": "V3D_balanced_train_split_70_15_15_f1fpr_added_benign_bruteforce",
        "threshold": th,
        "selection_objective": "val_f1_with_fpr_constraint",
        "final_status": final_status,
        "target_f1": TARGET_F1,
        "target_fpr": TARGET_FPR,
        "val_f1": float(best["val_f1"]),
        "val_fpr": float(best["val_fpr"]),
        "val_precision": float(best["val_precision"]),
        "val_recall": float(best["val_recall"]),
        "status_f1_ge_85": bool(m_test["f1"] >= TARGET_F1),
        "status_fpr_lt_2": bool(m_test["fpr"] < TARGET_FPR),
        "status_pass_both": bool((m_test["f1"] >= TARGET_F1) and (m_test["fpr"] < TARGET_FPR)),
    }
    row.update(m_test)
    final_rows.append(row)

    scen = scenario_table(test_df, s_test, th, model_name, "V3D_balanced_train_split_70_15_15_f1fpr_added_benign_bruteforce")
    scenario_parts.append(scen)

final_metrics_df = pd.DataFrame(final_rows)
final_scenario_df = pd.concat(scenario_parts, ignore_index=True)

display(final_metrics_df[[
    "model", "threshold", "final_status", "val_f1", "val_fpr",
    "f1", "fpr", "precision", "recall", "accuracy",
    "tn", "fp", "fn", "tp", "status_f1_ge_85", "status_fpr_lt_2", "status_pass_both"
]])
display(final_scenario_df)

final_metrics_df.to_csv(FINAL_TABLE_DIR / "BAB5_FINAL_V3D_BALANCED_SPLIT70_3model_test_metrics.csv", index=False)
final_scenario_df.to_csv(FINAL_TABLE_DIR / "BAB5_FINAL_V3D_BALANCED_SPLIT70_3model_by_scenario.csv", index=False)

thresholds_v3d_balanced_split70 = dict(zip(final_metrics_df["model"], final_metrics_df["threshold"]))
(FINAL_MODEL_DIR / "thresholds_final_v3d_balanced_train_split_70_15_15_f1fpr_added_benign_bruteforce.json").write_text(
    json.dumps(thresholds_v3d_balanced_split70, indent=2), encoding="utf-8"
)

print("Saved final metrics and thresholds.")
print(thresholds_v3d_balanced_split70)


,model,threshold,final_status,val_f1,val_fpr,f1,fpr,precision,recall,accuracy,tn,fp,fn,tp,status_f1_ge_85,status_fpr_lt_2,status_pass_both
0,AE-MLP,0.683,PASS_BOTH,0.848651,0.001866,0.910085,0.000373,0.999700,0.835214,0.901320,2680,1,657,3330,True,True,True
1,MLP,0.372,PASS_BOTH,0.887291,0.014179,0.944203,0.005595,0.995826,0.897667,0.936563,2666,15,408,3579,True,True,True
2,LightGBM,0.536,PASS_BOTH,0.905712,0.017910,0.952120,0.010817,0.992115,0.915224,0.944961,2652,29,338,3649,True,True,True


,model,version,scenario,ground_truth,n_flow,score_mean,score_min,score_max,pred_benign,pred_attack,fpr,detection_rate
0,AE-MLP,V3D_balanced_train_split_70_15_15_f1fpr_added_...,benign,0,2681,0.210582,4.920682e-06,0.690655,2680,1,0.000373,NaN
1,AE-MLP,V3D_balanced_train_split_70_15_15_f1fpr_added_...,bruteforce,1,499,0.402793,2.794478e-02,0.999047,410,89,NaN,0.178357
2,AE-MLP,V3D_balanced_train_split_70_15_15_f1fpr_added_...,slowhttp,1,476,0.890772,2.094251e-04,1.000000,28,448,NaN,0.941176
3,AE-MLP,V3D_balanced_train_split_70_15_15_f1fpr_added_...,slowloris,1,528,0.707083,1.827258e-01,0.999993,219,309,NaN,0.585227
4,AE-MLP,V3D_balanced_train_split_70_15_15_f1fpr_added_...,synflood,1,2484,0.999532,9.934241e-01,1.000000,0,2484,NaN,1.000000
5,MLP,V3D_balanced_train_split_70_15_15_f1fpr_added_...,benign,0,2681,0.097838,2.281898e-07,0.700393,2666,15,0.005595,NaN
6,MLP,V3D_balanced_train_split_70_15_15_f1fpr_added_...,bruteforce,1,499,0.648165,9.560691e-05,0.999991,200,299,NaN,0.599198
7,MLP,V3D_balanced_train_split_70_15_15_f1fpr_added_...,slowhttp,1,476,0.964485,1.173266e-02,1.000000,18,458,NaN,0.962185
8,MLP,V3D_balanced_train_split_70_15_15_f1fpr_added_...,slowloris,1,528,0.617958,1.220248e-01,1.000000,190,338,NaN,0.640152
9,MLP,V3D_balanced_train_split_70_15_15_f1fpr_added_...,synflood,1,2484,0.999762,9.990585e-01,1.000000,0,2484,NaN,1.000000


Saved final metrics and thresholds.
{'AE-MLP': 0.683, 'MLP': 0.372, 'LightGBM': 0.536}


In [18]:
# ============================================================
# CELL 17 — BEST MODEL RECOMMENDATION
# ============================================================
# Rekomendasi model akhir dipilih setelah threshold setiap model dikunci dari validation set.

passed = final_metrics_df[final_metrics_df["status_pass_both"]].copy()

if passed.empty:
    best = final_metrics_df.sort_values(["status_fpr_lt_2", "f1", "fpr"], ascending=[False, False, True]).iloc[0]
    recommendation = "Tidak ada model yang memenuhi F1>=85% dan FPR<2% secara bersamaan. Gunakan best trade-off atau tambah data."
else:
    best = passed.sort_values(["f1", "fpr"], ascending=[False, True]).iloc[0]
    recommendation = "Ada model yang memenuhi F1>=85% dan FPR<2% pada held-out live test."

best_df = pd.DataFrame([best.to_dict()])
best_df["recommendation"] = recommendation

display(best_df)
best_df.to_csv(FINAL_TABLE_DIR / "BAB5_FINAL_V3D_BALANCED_SPLIT70_best_model_recommendation.csv", index=False)

print(recommendation)
print("Best model:", best["model"], "threshold:", best["threshold"])


,model,version,threshold,selection_objective,final_status,target_f1,target_fpr,val_f1,val_fpr,val_precision,...,accuracy,precision,recall,f1,fpr,tn,fp,fn,tp,recommendation
0,LightGBM,V3D_balanced_train_split_70_15_15_f1fpr_added_...,0.536,val_f1_with_fpr_constraint,PASS_BOTH,0.85,0.02,0.905712,0.01791,0.985824,...,0.944961,0.992115,0.915224,0.95212,0.010817,2652,29,338,3649,Ada model yang memenuhi F1>=85% dan FPR<2% pad...


Ada model yang memenuhi F1>=85% dan FPR<2% pada held-out live test.
Best model: LightGBM threshold: 0.536


In [19]:
# ============================================================
# CELL 18 — BAB 5 READY SUMMARY TEXT
# ============================================================

def pct(x):
    if pd.isna(x):
        return "-"
    return f"{100*x:.2f}%"

lines = []
lines.append("RINGKASAN FINAL V3D BALANCED TRAIN SPLIT 70-15-15 — F1/FPR THRESHOLD")
lines.append("=" * 75)
lines.append("")
lines.append("Tujuan:")
lines.append(f"- Memenuhi target live binary detection: F1-score >= {TARGET_F1:.0%} dan FPR < {TARGET_FPR:.0%}.")
lines.append("- Menguji split 70:15:15 dengan data train seimbang benign:attack = 1:1.")
lines.append("")
lines.append("Metodologi:")
lines.append("- Data live runtime diambil dari *_ids_log.csv yang berisi 38 fitur runtime.")
lines.append("- Label live traffic ditentukan berdasarkan skenario pengujian, bukan berdasarkan prediksi model.")
lines.append("- Data dibagi secara time-ordered per skenario menjadi train, validation, dan test dengan rasio 70:15:15.")
lines.append("- Hanya data train yang diseimbangkan menjadi benign:attack = 1:1; validation dan test tetap natural.")
lines.append("- Threshold dipilih menggunakan validation set berdasarkan F1-score tertinggi dengan constraint FPR.")
lines.append("- Test set hanya digunakan untuk evaluasi akhir setelah threshold dikunci.")
lines.append("- Zero-day tetap dipisahkan melalui Infiltration holdout offline dari CSE-CIC-IDS2018.")
lines.append("")

lines.append("Hasil held-out live test per model:")
for _, r in final_metrics_df.iterrows():
    lines.append(
        f"- {r['model']}: threshold={r['threshold']:.3f}, "
        f"F1={pct(r['f1'])}, FPR={pct(r['fpr'])}, "
        f"Precision={pct(r['precision'])}, Recall={pct(r['recall'])}, "
        f"TP={int(r['tp'])}, FP={int(r['fp'])}, FN={int(r['fn'])}, TN={int(r['tn'])}, "
        f"PASS={bool(r['status_pass_both'])}"
    )

lines.append("")
lines.append("Rekomendasi:")
lines.append(str(recommendation))
lines.append(f"Model rekomendasi: {best['model']} dengan threshold {best['threshold']:.3f}.")

if bool(best["status_pass_both"]):
    lines.append("Model rekomendasi memenuhi target F1-score minimal 85% dan FPR di bawah 2% pada held-out live test.")
else:
    lines.append("Model rekomendasi belum memenuhi kedua target secara bersamaan. Perlu penambahan data atau kalibrasi lanjutan.")

lines.append("")
lines.append("Narasi aman untuk laporan:")
lines.append(
    "Pada eksperimen ini, rasio pembagian data diubah menjadi 70:15:15 dan data train diseimbangkan menjadi benign:attack = 1:1. "
    "Data validasi dan data uji tetap mengikuti distribusi live traffic asli agar proses kalibrasi threshold dan evaluasi akhir tetap natural. "
    "Threshold dipilih berdasarkan F1-score tertinggi pada data validasi dengan batasan FPR, sehingga data uji tidak digunakan dalam proses pemilihan threshold."
)

summary_text = "\n".join(lines)
print(summary_text)

summary_path = FINAL_DIR / "BAB5_READY_FINAL_V3D_BALANCED_SPLIT70_SUMMARY.txt"
summary_path.write_text(summary_text, encoding="utf-8")
print("\nSaved:", summary_path)


RINGKASAN FINAL V3D BALANCED TRAIN SPLIT 70-15-15 — F1/FPR THRESHOLD

Tujuan:
- Memenuhi target live binary detection: F1-score >= 85% dan FPR < 2%.
- Menguji split 70:15:15 dengan data train seimbang benign:attack = 1:1.

Metodologi:
- Data live runtime diambil dari *_ids_log.csv yang berisi 38 fitur runtime.
- Label live traffic ditentukan berdasarkan skenario pengujian, bukan berdasarkan prediksi model.
- Data dibagi secara time-ordered per skenario menjadi train, validation, dan test dengan rasio 70:15:15.
- Hanya data train yang diseimbangkan menjadi benign:attack = 1:1; validation dan test tetap natural.
- Threshold dipilih menggunakan validation set berdasarkan F1-score tertinggi dengan constraint FPR.
- Test set hanya digunakan untuk evaluasi akhir setelah threshold dikunci.
- Zero-day tetap dipisahkan melalui Infiltration holdout offline dari CSE-CIC-IDS2018.

Hasil held-out live test per model:
- AE-MLP: threshold=0.683, F1=91.01%, FPR=0.04%, Precision=99.97%, Recall=83.52%, 

## Output final

Folder output:

`/content/drive/MyDrive/CICIDS2018/ICICoS2026_FINAL_all_in_one_reviewer_compliant/FINAL_V3D_BALANCED_SPLIT70_added_benign_bruteforce_live_target/`

File penting:
- `models/ae_mlp_classifier_v3d_balanced_train_split_70_15_15_f1fpr_added_benign_bruteforce.keras`
- `models/ae_mlp_classifier_v3d_balanced_train_split_70_15_15_f1fpr_added_benign_bruteforce.tflite`
- `models/mlp_model_v3d_balanced_train_split_70_15_15_f1fpr_added_benign_bruteforce.keras`
- `models/mlp_model_v3d_balanced_train_split_70_15_15_f1fpr_added_benign_bruteforce.tflite`
- `models/lgbm_model_v3d_balanced_train_split_70_15_15_f1fpr_added_benign_bruteforce.txt`
- `models/thresholds_final_v3d_balanced_train_split_70_15_15_f1fpr_added_benign_bruteforce.json`
- `tables/BAB5_FINAL_V3D_BALANCED_SPLIT70_3model_test_metrics.csv`
- `tables/BAB5_FINAL_V3D_BALANCED_SPLIT70_3model_by_scenario.csv`
- `tables/BAB5_FINAL_V3D_BALANCED_SPLIT70_best_model_recommendation.csv`
- `BAB5_READY_FINAL_V3D_BALANCED_SPLIT70_SUMMARY.txt`

File ZIP live terbaru yang harus diletakkan di Drive:
`/content/drive/MyDrive/CICIDS2018/drive-download-20260629T134845Z-3-001.zip`
